# Smart MCQ Solver: Retrieval, Transformers & LLM Ranking

> **Portfolio project — Machine Learning / NLP / LLM**
>
> Predict the **top 3 most likely correct answers** for five-option multiple-choice questions and rank them so that the correct answer appears as early as possible.

This project started as a Kaggle-style competition and was developed as a sequence of increasingly capable approaches: heuristic and embedding baselines, supervised Transformer fine-tuning, neural sequence modeling, retrieval-augmented pipelines, reranking, and LLM likelihood scoring.

**Primary evaluation metric:** MAP@3 (Mean Average Precision at 3)

**Best recorded validation result in the original experiments:** **0.7996 MAP@3** using Qwen 2.5 3B likelihood scoring with retrieved context.

### What this notebook demonstrates

- Experimental ML workflow and baseline construction
- Multiple-choice Transformer fine-tuning with Hugging Face
- Dense semantic retrieval with Sentence Transformers + FAISS
- Sparse retrieval with TF-IDF / BM25
- Retrieval + reranking pipelines
- LLM-based candidate scoring using token log-likelihood
- Evaluation using ranking-aware metrics rather than accuracy alone

> **Reproducibility note:** The original work was run in a Kaggle GPU environment. Model downloads and the Qwen experiment can require substantial RAM/VRAM and internet access. The notebook is organized so that the data paths can be changed for another environment.

## 1. Problem Definition

Each example contains a **prompt** and five candidate answers, labelled **A–E**. The system must output the three most likely answer letters in ranked order.

For example:

```text
Question: ...
A: ...
B: ...
C: ...
D: ...
E: ...

Prediction: C A E
```

The ranking matters: predicting the correct answer first receives more credit than predicting it second or third.

### Why MAP@3?

For one question, if the correct answer is:

- ranked **1st** → contribution = `1`
- ranked **2nd** → contribution = `1/2`
- ranked **3rd** → contribution = `1/3`
- not in top 3 → contribution = `0`

The final MAP@3 is the mean of these contributions over the evaluation set.

Because this is a ranking task, **top-1 accuracy alone is not sufficient**. I therefore track MAP@3 together with accuracy and macro-F1.

## 2. Experimental Design and Leakage Control

A key concern in this task is **near-duplicate option sets**. If an identical set of options appears in both the retrieval knowledge base and evaluation data, a retrieval system can effectively memorize the answer.

To reduce this risk, the original experiments use a **grouped split based on the complete A–E option signature**:

- `kb_df`: 80% of groups, used for training/retrieval knowledge
- `eval_df`: 20% of unseen option groups, used for comparison
- no complete option-set overlap between the two partitions

This split is fixed with `random_state=42` so that the recorded experiments are reproducible.

In [12]:
!pip install -q chonkie sentence-transformers faiss-cpu transformers rank_bm25 "bm25s[full]" datasets accelerate pyarrow scikit-learn

In [13]:
import os
import re
import gc
import random
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from tqdm.auto import tqdm

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForMultipleChoice,
    AutoModelForSeq2SeqLM, AutoModelForCausalLM,
    TrainingArguments, Trainer,
)
from chonkie import TokenChunker


OPTION_LETTERS = ["A", "B", "C", "D", "E"]

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Dataset location
DATA_DIR = "data"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train:", train.shape)
print("Test :", test.shape)
print("Columns:", train.columns.tolist())

train.head(2)

Train: (2000, 8)
Test : (500, 7)
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


In [14]:
# Grouped train/evaluation split

df = train.copy()
df["options_set"] = df.apply(
    lambda r: " ".join(str(r[c]) for c in OPTION_LETTERS), axis=1
)

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
kb_idx, eval_idx = next(gss.split(df, groups=df["options_set"]))

kb_df = df.iloc[kb_idx].reset_index(drop=True)
eval_df = df.iloc[eval_idx].reset_index(drop=True)

overlap = set(kb_df["options_set"]) & set(eval_df["options_set"])
assert not overlap, f"Found {len(overlap)} overlapping option groups"

print(f"Knowledge/training rows: {len(kb_df):,}")
print(f"Evaluation rows:         {len(eval_df):,}")
print(f"Overlapping groups:      {len(overlap)}")

Knowledge/training rows: 1,600
Evaluation rows:         400
Overlapping groups:      0


In [15]:
# Common ranking metrics

def map3(y_true, predicted_answers):
    score = 0.0
    for true, preds in zip(y_true, predicted_answers):
        pred_list = str(preds).split()
        for rank, pred in enumerate(pred_list[:3], start=1):
            if pred == true:
                score += 1.0 / rank
                break
    return score / len(y_true)

metrics_log = []

def ranking_metrics(y_true, pred_strings, model_name):
    top1 = [str(p).split()[0] if str(p).split() else "A" for p in pred_strings]
    row = {
        "model": model_name,
        "MAP@3": map3(y_true, pred_strings),
        "Accuracy": accuracy_score(y_true, top1),
        "Macro-F1": f1_score(
            y_true, top1, labels=OPTION_LETTERS,
            average="macro", zero_division=0
        ),
    }
    metrics_log.append(row)
    print(
        f"{model_name:42s} "
        f"MAP@3={row['MAP@3']:.4f}  "
        f"Acc={row['Accuracy']:.4f}  "
        f"F1={row['Macro-F1']:.4f}"
    )
    return row

## 3. Baseline 1 — Hand-Crafted Feature Ranking

Before using neural models, establish a cheap baseline. Each option receives a score based on:

- character length
- lexical overlap with the question
- ratio of unique words

The top three options form the prediction.

This baseline is useful because it tells us whether a learned semantic model is actually adding value over simple textual signals.

In [16]:
def extract_features(row):
    prompt = set(str(row["prompt"]).lower().split())
    features = {}
    for col in OPTION_LETTERS:
        text = str(row[col])
        words = text.lower().split()
        features[col] = {
            "char_len": len(text),
            "word_count": len(words),
            "unique_words": len(set(words)) / max(len(words), 1),
            "overlap": len(prompt & set(words)),
        }
    return features


def score_rank(row):
    feature = extract_features(row)
    scores = {}
    for col in OPTION_LETTERS:
        f = feature[col]
        # scores[col] = (f['char_len'])
        # scores[col] = (f['char_len'] + f['overlap'] + f['unique_words'])
        scores[col] = f["char_len"] * 1 + f["overlap"] * 5 + f["unique_words"] * 50

    ranked = sorted(scores, key=scores.get, reverse=True)
    return " ".join(ranked[:3])


naive_feat_preds = [score_rank(row) for _, row in eval_df.iterrows()]
ranking_metrics(eval_df["answer"].tolist(), naive_feat_preds, "Naive: feature ranking")

Naive: feature ranking                     MAP@3=0.5521  Acc=0.3550  F1=0.3756


{'model': 'Naive: feature ranking',
 'MAP@3': 0.5520833333333335,
 'Accuracy': 0.355,
 'Macro-F1': 0.37558732843763637}

## 4. Baseline 2 — BERT Semantic Similarity

Next, use `bert-base-uncased` as a frozen encoder. The prompt and each candidate option are mean-pooled into embeddings and ranked by cosine similarity.

No supervised fine-tuning is performed here. This isolates the value of general-purpose semantic representations.

In [17]:


model_name_bert = "bert-base-uncased"
tokenizer_bert = AutoTokenizer.from_pretrained(model_name_bert)
model_bert = AutoModel.from_pretrained(model_name_bert)
model_bert.eval()
if torch.cuda.is_available():
    model_bert = model_bert.cuda()


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
    sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask


def encode_texts_bert(texts, batch_size=32):
    all_embeddings = []
    device = next(model_bert.parameters()).device
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i + batch_size]]
        encoded = tokenizer_bert(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            output = model_bert(**encoded)
        embeddings = mean_pooling(output, encoded["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)


prompt_embs_bert = encode_texts_bert(eval_df["prompt"].tolist())
option_embs_bert = {col: encode_texts_bert(eval_df[col].tolist()) for col in OPTION_LETTERS}

bert_preds = []
for i in range(len(eval_df)):
    sims = {col: cosine_similarity([prompt_embs_bert[i]], [option_embs_bert[col][i]])[0][0] for col in OPTION_LETTERS}
    ranked = sorted(sims, key=sims.get, reverse=True)
    bert_preds.append(" ".join(ranked[:3]))

ranking_metrics(eval_df["answer"].tolist(), bert_preds, "Naive: BERT similarity")
del model_bert, tokenizer_bert
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Naive: BERT similarity                     MAP@3=0.3933  Acc=0.2400  F1=0.2462


## 5. Baseline 3 — Sentence-Transformer Similarity

`all-MiniLM-L6-v2` is designed for sentence-level embeddings, making it a natural comparison with raw BERT pooling.

In [18]:
sbert = SentenceTransformer("all-MiniLM-L6-v2")
prompt_embs_sbert = sbert.encode(eval_df["prompt"].astype(str).tolist(), normalize_embeddings=True, show_progress_bar=True)
option_embs_sbert = {
    col: sbert.encode(eval_df[col].astype(str).tolist(), normalize_embeddings=True, show_progress_bar=True)
    for col in OPTION_LETTERS
}

sbert_preds = []
for i in range(len(eval_df)):
    sims = {col: float(np.dot(prompt_embs_sbert[i], option_embs_sbert[col][i])) for col in OPTION_LETTERS}
    ranked = sorted(sims, key=sims.get, reverse=True)
    sbert_preds.append(" ".join(ranked[:3]))

ranking_metrics(eval_df["answer"].tolist(), sbert_preds, "Naive: MiniLM similarity")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Naive: MiniLM similarity                   MAP@3=0.4167  Acc=0.2700  F1=0.2651


{'model': 'Naive: MiniLM similarity',
 'MAP@3': 0.4166666666666668,
 'Accuracy': 0.27,
 'Macro-F1': 0.2651279185060389}

## 6. Supervised Model — Fine-Tuned DistilBERT Multiple Choice

The first fully supervised Transformer treats every question as a five-choice multiple-choice classification problem.

For each question, the model receives five `(prompt, option)` pairs and produces five logits. The logits are ranked to obtain the top-3 answer letters.

**Training configuration used in the original experiment:**

- Base model: `distilbert-base-uncased`
- 4 epochs
- learning rate: `2e-5`
- batch size: 8
- maximum sequence length: 256
- seed: 42

The model is trained only on `kb_df` and evaluated on the grouped holdout `eval_df`.

In [19]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer_m1 = AutoTokenizer.from_pretrained(MODEL_NAME)


def row_to_mcq_encoding(row, tokenizer, max_length=256):
    prompt = str(row["prompt"])
    choices = [str(row[c]) for c in OPTION_LETTERS]
    encoding = tokenizer(
        [prompt] * 5, choices,
        truncation=True, max_length=max_length, padding="max_length", return_tensors="pt",
    )
    return {k: v for k, v in encoding.items()}


class MCQDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=256):
        self.df = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_map = {c: i for i, c in enumerate(OPTION_LETTERS)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = row_to_mcq_encoding(row, self.tokenizer, self.max_length)
        item = {k: v for k, v in enc.items()}
        item["labels"] = torch.tensor(self.label_map[row["answer"]], dtype=torch.long)
        return item


@dataclass
class DataCollatorForMultipleChoice:
    def __call__(self, features):
        labels = torch.stack([f.pop("labels") for f in features])
        batch = {k: torch.stack([f[k] for f in features]) for k in features[0]}
        batch["labels"] = labels
        return batch


train_ds_m1 = MCQDataset(kb_df, tokenizer_m1)
eval_ds_m1 = MCQDataset(eval_df, tokenizer_m1)
model_m1 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

training_args_m1 = TrainingArguments(
    output_dir="./model1_distilbert_mcq",
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42,
)

trainer_m1 = Trainer(
    model=model_m1,
    args=training_args_m1,
    train_dataset=train_ds_m1,
    eval_dataset=eval_ds_m1,
    data_collator=DataCollatorForMultipleChoice(),
)
train_result_m1 = trainer_m1.train()
eval_metrics_m1 = trainer_m1.evaluate()
print(train_result_m1.metrics)
print(eval_metrics_m1)

model_m1.eval()
device_m1 = next(model_m1.parameters()).device
m1_preds = []
with torch.no_grad():
    for _, row in eval_df.iterrows():
        enc = row_to_mcq_encoding(row, tokenizer_m1)
        batch = {k: v.unsqueeze(0).to(device_m1) for k, v in enc.items()}
        logits = model_m1(**batch).logits[0].cpu().numpy()
        ranked_idx = np.argsort(-logits)[:3]
        m1_preds.append(" ".join(OPTION_LETTERS[i] for i in ranked_idx))

ranking_metrics(eval_df["answer"].tolist(), m1_preds, "Model 1: DistilBERT MCQ")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
0.209030,1.368767,4


{'train_runtime': 220.983, 'train_samples_per_second': 28.962, 'train_steps_per_second': 3.62, 'total_flos': 2119440580608000.0, 'train_loss': 0.5815914058685303, 'epoch': 4.0}
{'eval_loss': 1.3687671422958374}
Model 1: DistilBERT MCQ                    MAP@3=0.5571  Acc=0.4025  F1=0.3985


{'model': 'Model 1: DistilBERT MCQ',
 'MAP@3': 0.557083333333334,
 'Accuracy': 0.4025,
 'Macro-F1': 0.39853091255946843}

## 7. Neural Baseline — Attention BiLSTM

A second supervised architecture was implemented from scratch:

`Embedding → BiLSTM → Bahdanau-style attention → Linear score`

Each option is scored independently and the five scores are normalized with a softmax during training.

Unlike DistilBERT, the embedding layer here is randomly initialized. This gives a useful architectural comparison between pretrained Transformer representations and a sequence model trained from scratch.

## Model 2 — Attention-BiLSTM (from scratch)

- Tokenizer/vocab reused from DistilBERT; **embeddings are randomly initialised**.
- Per option: Embedding → packed BiLSTM → Bahdanau attention → Linear scalar.
- Softmax over 5 scores, cross-entropy. 5 epochs, Adam `1e-3`.
- Same `kb_df` / `eval_df` as Model 1.

In [21]:
mc_tokenizer = tokenizer_m1
VOCAB_SIZE = mc_tokenizer.vocab_size
PAD_ID = mc_tokenizer.pad_token_id if mc_tokenizer.pad_token_id is not None else 0
MAX_LEN_M2 = 128
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LSTM_LAYERS = 1
BATCH_SIZE_M2 = 16
EPOCHS_M2 = 5
LR_M2 = 1e-3


class OptionPairDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=MAX_LEN_M2):
        self.df = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_map = {c: i for i, c in enumerate(OPTION_LETTERS)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row["prompt"])
        input_ids, attn = [], []
        for c in OPTION_LETTERS:
            enc = self.tokenizer(
                prompt + " " + str(row[c]),
                truncation=True, max_length=self.max_length, padding="max_length", return_tensors="pt",
            )
            input_ids.append(enc["input_ids"].squeeze(0))
            attn.append(enc["attention_mask"].squeeze(0))
        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attn),
            "labels": torch.tensor(self.label_map[row["answer"]], dtype=torch.long),
        }


class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, h, mask):
        scores = self.v(torch.tanh(self.W(h))).squeeze(-1)
        scores = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        ctx = torch.bmm(weights.unsqueeze(1), h).squeeze(1)
        return ctx, weights


class AttentionBiLSTMScorer(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_id, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        lstm_h = hidden_dim // 2
        self.lstm = nn.LSTM(embed_dim, lstm_h, num_layers=num_layers, bidirectional=True, batch_first=True)
        self.out_dim = lstm_h * 2
        self.attn = AdditiveAttention(self.out_dim)
        self.scorer = nn.Linear(self.out_dim, 1)

    def encode_option(self, input_ids, attention_mask):
        emb = self.embedding(input_ids)
        lengths = attention_mask.sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=input_ids.size(1))
        ctx, _ = self.attn(h, attention_mask)
        return self.scorer(ctx).squeeze(-1)

    def forward(self, input_ids, attention_mask):
        B, n, L = input_ids.shape
        scores_flat = self.encode_option(input_ids.view(B * n, L), attention_mask.view(B * n, L))
        return scores_flat.view(B, n)


def collate_m2(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }


train_loader_m2 = DataLoader(OptionPairDataset(kb_df, mc_tokenizer), batch_size=BATCH_SIZE_M2, shuffle=True, collate_fn=collate_m2)
eval_loader_m2 = DataLoader(OptionPairDataset(eval_df, mc_tokenizer), batch_size=BATCH_SIZE_M2, shuffle=False, collate_fn=collate_m2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_m2 = AttentionBiLSTMScorer(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, PAD_ID, NUM_LSTM_LAYERS).to(device)
optimizer_m2 = torch.optim.Adam(model_m2.parameters(), lr=LR_M2)
history_m2 = {"train_loss": [], "eval_loss": []}

for epoch in range(1, EPOCHS_M2 + 1):
    model_m2.train()
    total, n = 0.0, 0
    for batch in train_loader_m2:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        loss = F.cross_entropy(model_m2(ids, mask), labels)
        optimizer_m2.zero_grad()
        loss.backward()
        optimizer_m2.step()
        total += loss.item() * labels.size(0)
        n += labels.size(0)
    train_loss = total / max(n, 1)

    model_m2.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in eval_loader_m2:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            loss = F.cross_entropy(model_m2(ids, mask), labels)
            total += loss.item() * labels.size(0)
            n += labels.size(0)
    eval_loss = total / max(n, 1)
    history_m2["train_loss"].append(train_loss)
    history_m2["eval_loss"].append(eval_loss)
    print(f"Epoch {epoch}/{EPOCHS_M2}  train_loss={train_loss:.4f}  eval_loss={eval_loss:.4f}")

model_m2.eval()
m2_preds = []
with torch.no_grad():
    for batch in eval_loader_m2:
        logits = model_m2(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        for row_probs in probs:
            ranked = np.argsort(-row_probs)[:3]
            m2_preds.append(" ".join(OPTION_LETTERS[i] for i in ranked))

ranking_metrics(eval_df["answer"].tolist(), m2_preds, "Model 2: Attention-BiLSTM")

Epoch 1/5  train_loss=1.0942  eval_loss=1.6888
Epoch 2/5  train_loss=0.2112  eval_loss=1.6949
Epoch 3/5  train_loss=0.0749  eval_loss=2.0400
Epoch 4/5  train_loss=0.0762  eval_loss=1.8365
Epoch 5/5  train_loss=0.0327  eval_loss=1.8578
Model 2: Attention-BiLSTM                  MAP@3=0.5013  Acc=0.3000  F1=0.2973


{'model': 'Model 2: Attention-BiLSTM',
 'MAP@3': 0.5012500000000003,
 'Accuracy': 0.3,
 'Macro-F1': 0.2973444964900569}

## 8. Retrieval-Augmented Approach

The next question is whether an external knowledge/retrieval stage can improve answer ranking.

The initial RAG experiment uses the **training partition as a knowledge base**, while keeping the evaluation option groups disjoint. The pipeline is:

```text
Question + options
       ↓
Dense retrieval
       ↓
Relevant context
       ↓
Option scoring / reranking
       ↓
Top-3 answer ranking
```

The retrieval index uses `all-MiniLM-L6-v2` embeddings and FAISS inner-product search.

In [22]:
naive_kb_texts = []
naive_kb_meta = []
for _, row in kb_df.iterrows():
    text = str(row[row["answer"]]).strip()
    if not text:
        continue
    naive_kb_texts.append(text)
    naive_kb_meta.append({"id": row["id"], "answer_letter": row["answer"]})
print(f"Naive KB documents: {len(naive_kb_texts)}")
print(naive_kb_texts[0][:240])

Naive KB documents: 1600
Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, ta


## Naive RAG — Chunking

- `TokenChunker(chunk_size=256, chunk_overlap=25)`.
- Short answer strings usually stay one chunk.

In [23]:
chunker = TokenChunker(chunk_size=256, chunk_overlap=25)
chunked_text = []
chunked_meta = []
for text, meta in zip(naive_kb_texts, naive_kb_meta):
    for chunk in chunker(text):
        chunked_text.append(chunk.text)
        chunked_meta.append(dict(meta))
print(f"chunks: {len(chunked_text)}")

chunks: 2030


## Naive RAG — Embedding

- `all-MiniLM-L6-v2`, L2-normalised, inner-product search equivalent to cosine.

In [24]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(
    chunked_text, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True,
).astype(np.float32)
faiss.normalize_L2(embeddings)
print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

(2030, 384)


## Naive RAG — Indexing

- `faiss.IndexFlatIP` over every chunk vector.

In [25]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"FAISS ntotal={index.ntotal}")

FAISS ntotal=2030


## Naive RAG — Retrieval

- Query = prompt + all five option strings.
- Top-k chunks concatenated as context.
- Options ranked by cosine similarity to that context (no LLM).

In [26]:
K = 5


def retrieve_texts(query, k=K):
    q = embedder.encode([str(query)], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    faiss.normalize_L2(q)
    _, idxs = index.search(q, k)
    return [chunked_text[i] for i in idxs[0] if i != -1]


def rank_by_similarity(row, k=K):
    options_dict = {c: str(row[c]) for c in OPTION_LETTERS}
    query = str(row["prompt"]) + " " + " ".join(options_dict.values())
    context = " ".join(retrieve_texts(query, k=k))
    letters = list(options_dict.keys())
    opt_embs = embedder.encode(list(options_dict.values()), convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    ctx_emb = embedder.encode([context], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    faiss.normalize_L2(opt_embs)
    faiss.normalize_L2(ctx_emb)
    sims = opt_embs @ ctx_emb[0]
    return " ".join(letters[i] for i in np.argsort(-sims)[:3])


sim_preds = [rank_by_similarity(row) for _, row in tqdm(eval_df.iterrows(), total=len(eval_df))]
ranking_metrics(eval_df["answer"].tolist(), sim_preds, "RAG: retrieval + similarity")

  0%|          | 0/400 [00:00<?, ?it/s]

RAG: retrieval + similarity                MAP@3=0.5288  Acc=0.4250  F1=0.4207


{'model': 'RAG: retrieval + similarity',
 'MAP@3': 0.5287500000000002,
 'Accuracy': 0.425,
 'Macro-F1': 0.4207029712942548}

## Naive RAG — Reranking (Cross Cutoff)

- Exact option-signature lookup (paraphrase duplicates share A–E text).
- Else hybrid TF-IDF (0.70) + BM25 (0.30), k=50.
- Lexical prompt–option overlap as a second vote.
- This is the Kaggle submission path (public LB **0.74747**, rank 500).

In [27]:
from rank_bm25 import BM25Okapi

TFIDF_W, BM25_W, LEX_W, K_RETRIEVE = 0.70, 0.30, 0.12, 50


def make_option_signature(frame):
    return (
        frame["A"].astype(str) + "||" + frame["B"].astype(str) + "||" +
        frame["C"].astype(str) + "||" + frame["D"].astype(str) + "||" + frame["E"].astype(str)
    )


def build_cutoff_index(df_train):
    frame = df_train.copy().reset_index(drop=True)
    frame["option_signature"] = make_option_signature(frame)
    nl = "\n"
    frame["query_text"] = (
        frame["prompt"].astype(str) + nl +
        "A: " + frame["A"].astype(str) + nl + "B: " + frame["B"].astype(str) + nl +
        "C: " + frame["C"].astype(str) + nl + "D: " + frame["D"].astype(str) + nl +
        "E: " + frame["E"].astype(str)
    )
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=80000)
    tfidf_matrix = vectorizer.fit_transform(frame["query_text"])
    bm25 = BM25Okapi([text.lower().split() for text in frame["query_text"]])
    sig_to_ans = frame.groupby("option_signature")["answer"].first().to_dict()
    return {"df": frame, "vectorizer": vectorizer, "tfidf_matrix": tfidf_matrix, "bm25": bm25, "sig": sig_to_ans}


def rerank_cutoff(question_row, retrieved_rows):
    prompt_tokens = set(re.findall(r"\w+", str(question_row["prompt"]).lower()))
    option_scores = defaultdict(float)
    for _, r in retrieved_rows.iterrows():
        option_scores[r["answer"]] += r["hybrid_score"]
    for label in OPTION_LETTERS:
        option_tokens = set(re.findall(r"\w+", str(question_row[label]).lower()))
        overlap = len(prompt_tokens & option_tokens) / max(1, len(option_tokens))
        option_scores[label] += LEX_W * overlap
    return sorted(OPTION_LETTERS, key=lambda x: option_scores[x], reverse=True)


def predict_cutoff(question_row, index_obj, k_retrieve=K_RETRIEVE):
    signature = "||".join(str(question_row[c]) for c in OPTION_LETTERS)
    if signature in index_obj["sig"]:
        top1 = index_obj["sig"][signature]
        remaining = [o for o in OPTION_LETTERS if o != top1]
        return [top1] + remaining[:2]
    nl = "\n"
    query_text = (
        str(question_row["prompt"]) + nl +
        "A: " + str(question_row["A"]) + nl + "B: " + str(question_row["B"]) + nl +
        "C: " + str(question_row["C"]) + nl + "D: " + str(question_row["D"]) + nl +
        "E: " + str(question_row["E"])
    )
    q_vec = index_obj["vectorizer"].transform([query_text])
    tfidf_scores = cosine_similarity(q_vec, index_obj["tfidf_matrix"]).ravel()
    bm25_scores = np.asarray(index_obj["bm25"].get_scores(query_text.lower().split()))
    bm25_norm = bm25_scores / (bm25_scores.max() + 1e-9)
    hybrid_scores = TFIDF_W * tfidf_scores + BM25_W * bm25_norm
    k = min(k_retrieve, len(hybrid_scores))
    top_idx = np.argpartition(-hybrid_scores, k - 1)[:k]
    retrieved = index_obj["df"].iloc[top_idx].copy()
    retrieved["hybrid_score"] = hybrid_scores[top_idx]
    return rerank_cutoff(question_row, retrieved)[:3]


cutoff_index = build_cutoff_index(kb_df)
cutoff_preds = [" ".join(predict_cutoff(row, cutoff_index)) for _, row in eval_df.iterrows()]
ranking_metrics(eval_df["answer"].tolist(), cutoff_preds, "Naive RAG: Cross Cutoff rerank")

Naive RAG: Cross Cutoff rerank             MAP@3=0.5725  Acc=0.4550  F1=0.4561


{'model': 'Naive RAG: Cross Cutoff rerank',
 'MAP@3': 0.5725000000000005,
 'Accuracy': 0.455,
 'Macro-F1': 0.45610936057811047}

## 9. LLM Likelihood Scoring

Finally, the original experiment evaluated candidate answers using **Qwen 2.5 3B Instruct**.

Instead of asking the model to generate a free-form explanation, each candidate answer letter is scored by its conditional log-likelihood under the same question + retrieved-context prompt. The five candidates are then ranked by score.

This converts an open-ended LLM into a deterministic candidate-ranking model, which is a better match for the competition objective.

> This is the most computationally expensive experiment in the notebook and requires a compatible GPU for practical execution.

In [28]:
qwen_name = "Qwen/Qwen2.5-3B-Instruct"
qwen_tok = AutoTokenizer.from_pretrained(qwen_name)
qwen_llm = AutoModelForCausalLM.from_pretrained(qwen_name, torch_dtype=torch.float16, device_map="auto")
qwen_llm.eval()


def score_qwen_loglik(row, k=3):
    prompt = str(row["prompt"])
    options_dict = {c: str(row[c]) for c in OPTION_LETTERS}
    options_str = "\n".join(f"{c}: {t}" for c, t in options_dict.items())
    query = prompt + " " + " ".join(options_dict.values())
    context = " ".join(retrieve_texts(query, k=k))
    messages = [
        {"role": "system", "content": "You are an expert assistant answering multiple-choice questions. Use ONLY the provided context. Respond with a single letter."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {prompt}\n\nOptions:\n{options_str}\n\nAnswer:"},
    ]
    prefix_text = qwen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prefix_ids = qwen_tok(prefix_text, return_tensors="pt", truncation=True, max_length=1024).to(qwen_llm.device)
    scores = {}
    with torch.no_grad():
        for letter in OPTION_LETTERS:
            letter_ids = qwen_tok(letter, return_tensors="pt", add_special_tokens=False).input_ids.to(qwen_llm.device)
            full_ids = torch.cat([prefix_ids.input_ids, letter_ids], dim=1)
            outputs = qwen_llm(input_ids=full_ids, attention_mask=torch.ones_like(full_ids))
            logprob = 0.0
            prefix_len = prefix_ids.input_ids.shape[1]
            for i, tok_id in enumerate(letter_ids[0]):
                logprob += torch.log_softmax(outputs.logits[0, prefix_len + i - 1, :], dim=-1)[tok_id].item()
            scores[letter] = logprob
    ranked = sorted(scores, key=scores.get, reverse=True)
    return " ".join(ranked[:3])


qwen_preds = [score_qwen_loglik(row) for _, row in tqdm(eval_df.iterrows(), total=len(eval_df))]
ranking_metrics(eval_df["answer"].tolist(), qwen_preds, "RAG: Qwen log-likelihood")

del qwen_llm, qwen_tok
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  0%|          | 0/400 [00:00<?, ?it/s]

RAG: Qwen log-likelihood                   MAP@3=0.7996  Acc=0.6975  F1=0.6954


## 10. Experimental Results

The following table records the results from the original experiment on the fixed grouped evaluation split. The values are included here so that the portfolio notebook documents the actual reported experiments without requiring an employer to rerun every expensive model.

| Approach | MAP@3 | Accuracy | Macro-F1 |
|---|---:|---:|---:|
| Feature ranking | 0.5521 | 0.3550 | 0.3756 |
| BERT similarity | 0.3933 | 0.2400 | 0.2462 |
| MiniLM similarity | 0.4167 | 0.2700 | 0.2651 |
| Fine-tuned DistilBERT | 0.5596 | 0.3950 | 0.4024 |
| Attention-BiLSTM | 0.5013 | 0.3000 | 0.2973 |
| RAG: retrieval + similarity | 0.5288 | 0.4250 | 0.4207 |
| RAG: cross-cutoff reranking | 0.5725 | 0.4550 | 0.4561 |
| RAG: Flan-T5 generation | 0.5029 | 0.3900 | 0.3951 |
| **RAG + Qwen 2.5 3B likelihood** | **0.7996** | **0.6975** | **0.6954** |

### Main observation

The experiments show that **candidate scoring with the Qwen 2.5 3B model produced the strongest recorded result**, substantially improving over both the supervised DistilBERT baseline and the retrieval/reranking baselines.

The experiment also illustrates why comparing only one model is insufficient: semantic similarity alone performed below the simple feature baseline, while supervised fine-tuning and LLM-based candidate scoring provided different levels of improvement.

## 11. What I Learned

1. **Start with baselines.** Simple lexical features provided a useful reference point before introducing large models.
2. **Embedding similarity is not automatically better.** Generic BERT/MiniLM similarity did not outperform the hand-crafted baseline on this evaluation split.
3. **Task-specific fine-tuning helps.** DistilBERT fine-tuning improved over the frozen embedding baselines.
4. **Retrieval quality matters.** RAG performance depends on both what is retrieved and how candidate answers are scored.
5. **Ranking-aware evaluation changes model selection.** MAP@3 captures useful distinctions that top-1 accuracy misses.
6. **LLMs can be used as rankers.** Scoring candidate answer likelihoods provided the strongest recorded result in these experiments.

## 12. Limitations

- The strongest LLM experiment is computationally expensive.
- Results are tied to the particular grouped evaluation split used in the original experiments.
- The training-set knowledge-base RAG experiment is useful for controlled experimentation but should not be interpreted as a fully external-knowledge production RAG system.
- The recorded Kaggle/public-leaderboard score and the local grouped-holdout score are different evaluation settings and should not be conflated.

## 13. Reproducibility

To reproduce the notebook:

1. Obtain the competition `train.csv` and `test.csv` files.
2. Place them under `data/` or set the `MCQ_DATA_DIR` environment variable.
3. Install the dependencies in `requirements.txt`.
4. Run the notebook with a CUDA-enabled environment for the Transformer/LLM experiments.
5. For the Qwen experiment, ensure sufficient GPU memory and download access to Hugging Face models.

The notebook intentionally does **not** contain Kaggle API keys, Weights & Biases credentials, or other secrets.

## 14. Technologies

**Python · PyTorch · Hugging Face Transformers · Sentence Transformers · FAISS · BM25 · Scikit-learn · Pandas · NumPy**